# RAG Retrieval: Bi-Encoder Baseline vs. Cross-Encoder Re-Ranking

Two retrieval pipelines over the **same chunked corpus**, compared on stress-test questions:

- **Pipeline 1 (baseline):** bi-encoder embeddings in a vector store → top 10 by cosine similarity → top 3 to the LLM → *Answer 1*.
- **Pipeline 2 (re-ranking):** the same top 10 → cross-encoder scores each (query, chunk) pair → re-sorted → top 3 to the LLM with the identical prompt → *Answer 2*.

**Document:** NIST SP 800-63B-4, *Digital Identity Guidelines: Authentication and Authenticator Management* (July 2025, 129 pages, [doi:10.6028/NIST.SP.800-63B-4](https://doi.org/10.6028/NIST.SP.800-63B-4)). It is a technical policy made of normative `SHALL` / `SHALL NOT` rules, exceptions written as "unless…" or "however…", and informative sections (usability, appendices) that discuss the same topics in looser language.

| Component | Choice |
|---|---|
| PDF text extraction | `pypdf` |
| Chunking | sentence-aware, ~200 word-piece tokens, ~40-token overlap |
| Bi-encoder | `sentence-transformers/all-MiniLM-L6-v2` (384-dim, normalised → cosine) |
| Vector store | ChromaDB, in-memory collection with cosine distance |
| Cross-encoder | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | `openai/gpt-oss-120b` on the Groq API, temperature 0, same prompt for both pipelines |

**Contents:** 1 Setup · 2 Load the PDF · 3 Chunking · 4 Pipeline 1 · 5 Pipeline 2 · 6 Answer generation · 7 Running both pipelines · 8 Failure-mode analysis · 9 Cases re-ranking did not fix · 10 Latency profiling · 11 Saved results

## 1. Setup

Requires Python 3.12 and `pip install -r requirements.txt`. The two models are downloaded from the Hugging Face Hub on first use. Answer generation needs a (free) Groq API key in the `GROQ_API_KEY` environment variable; without one, every retrieval and re-ranking step still runs and the answers are marked as skipped.

In [1]:
import bisect
import os
import platform
import re
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm notice: ipywidgets is not installed

import chromadb
import numpy as np
import pandas as pd
import torch
from groq import Groq
from IPython.display import Markdown, display
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()  # keeps model-loading progress bars out of the saved outputs
hf_logging.set_verbosity_error()  # the tokenizer warns about >256-token inputs, but here they are only counted
pd.set_option("display.max_colwidth", 110)

PDF_PATH = Path("data/NIST.SP.800-63B-4.pdf")
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "openai/gpt-oss-120b"  # served by Groq

TOP_K = 10  # candidates retrieved by the bi-encoder
TOP_N = 3  # chunks passed to the LLM
CHUNK_TOKENS = 200  # all-MiniLM-L6-v2 truncates its input at 256 word pieces, so chunks stay below that
OVERLAP_TOKENS = 40  # about one or two sentences repeated at the start of the next chunk

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def cpu_name() -> str:
    try:  # Linux
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor() or platform.machine()


print(f"Python {platform.python_version()} | torch {torch.__version__} | chromadb {chromadb.__version__}")
print(f"CPU: {cpu_name()} | torch threads: {torch.get_num_threads()} | GPU used: no (all models run on CPU)")

Python 3.12.14 | torch 2.12.0+cpu | chromadb 1.5.9
CPU: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz | torch threads: 4 | GPU used: no (all models run on CPU)


## 2. Load and clean the PDF

`pypdf` extracts the text page by page. Every page except the first four repeats a four-line running header and ends with its page number, so both are removed. Words hyphenated across a line break are re-joined, and the dot leaders of the table of contents are dropped. Nothing else is filtered out: the table of contents, glossary and reference list stay in the corpus, as they would in a straightforward ingestion pipeline.

In [2]:
RUNNING_HEADER = "NIST SP 800-63B-4\nJuly 2025\nDigital Identity Guidelines\nAuthentication and Authenticator Management\n"


def clean_page(text: str) -> str:
    text = text.replace(RUNNING_HEADER, "")
    text = re.sub(r"\n\s*\d+\s*$", "", text)  # page number at the bottom of the page
    text = re.sub(r"-\n(?=\w)", "-", text)  # "multi-\nfactor" -> "multi-factor"
    text = re.sub(r"(?: ?\.){4,}", " ", text)  # ". . . . ." dot leaders in the table of contents
    return re.sub(r"\s+", " ", text).strip()


pages = [clean_page(page.extract_text()) for page in PdfReader(PDF_PATH).pages]
print(f"{len(pages)} pages, {sum(len(p.split()) for p in pages):,} words\n")
print("PDF page 25 (start):", pages[24][:700])

129 pages, 38,748 words

PDF page 25 (start): unlock passwords or PINs on many devices) are not sent to the verifier and instead used locally to gain access to the authentication secret. Passwords are not phishing-resistant. 3.1.1.1. Password Authenticators Passwords SHALL either be chosen by the subscriber or assigned randomly by the CSP . If the CSP disallows a chosen password because it is on a blocklist of commonly used, expected, or compromised values (see Sec. 3.1.1.2), the subscriber SHALL be required to choose a different password. Other composition requirements for passwords SHALL NOT be imposed. A rationale for this is presented in Appendix A, Strength of Passwords. 3.1.1.2. Password Verifiers The following requirements apply 


## 3. Chunking

Both pipelines search exactly this set of chunks.

1. The cleaned pages are joined into one string (so a sentence that crosses a page break stays whole), remembering the character offset where each page starts.
2. The text is split into sentences (after `.` `!` `?` when a new sentence follows) and before list bullets.
3. Sentences are packed greedily: a chunk is closed when the next sentence would take it past `CHUNK_TOKENS` word-piece tokens (counted with the bi-encoder's own tokenizer). The next chunk starts with the last sentences, up to `OVERLAP_TOKENS` tokens, of the previous one, so a rule and the sentence right after it are less likely to be separated. That carried-over overlap can push a chunk a little past `CHUNK_TOKENS`; the assert below checks that every chunk still fits in the bi-encoder's 256-token input.
4. A "sentence" longer than a whole chunk (mostly tables flattened by the PDF extraction) is cut into equal word windows first.

Each chunk records the PDF page on which it starts; the page is used for citations.

In [3]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")
tokenizer = bi_encoder.tokenizer


def n_tokens(text: str) -> int:
    return len(tokenizer.tokenize(text))


# 1) One string for the whole document + the character offset where each page starts.
full_text, page_starts = "", []
for page in pages:
    page_starts.append(len(full_text))
    full_text += page + " "

# 2) Sentence-like units with their page number and token count.
SENTENCE_BREAK = re.compile(r"(?<=[.!?])\s+(?=[A-Z•(\[“\d])|\s+(?=• )")
units, start = [], 0  # (text, page, tokens)
for match in [*SENTENCE_BREAK.finditer(full_text), None]:
    end = match.start() if match else len(full_text)
    sentence = full_text[start:end].strip()
    if sentence:
        page = bisect.bisect_right(page_starts, start)  # 1-based PDF page where the sentence starts
        words = sentence.split()
        parts = max(1, -(-n_tokens(sentence) // CHUNK_TOKENS))  # ceiling division; > 1 only for run-on tables
        size = -(-len(words) // parts)
        for i in range(0, len(words), size):
            piece = " ".join(words[i : i + size])
            units.append((piece, page, n_tokens(piece)))
    start = match.end() if match else end

# 3) Greedy packing with overlap.
chunks, current = [], []
for unit in units:
    if current and sum(u[2] for u in current) + unit[2] > CHUNK_TOKENS:
        chunks.append(current)
        overlap = []
        for u in reversed(current):  # carry the last sentences of this chunk into the next one
            if sum(x[2] for x in overlap) + u[2] > OVERLAP_TOKENS:
                break
            overlap.insert(0, u)
        current = overlap
    current.append(unit)
chunks.append(current)

chunks_df = pd.DataFrame({"page": [c[0][1] for c in chunks], "text": [" ".join(u[0] for u in c) for c in chunks]})
chunks_df["tokens"] = chunks_df.text.map(n_tokens)
chunks_df.index.name = "chunk_id"

# Self-check: the bi-encoder must see every chunk in full (its limit includes the [CLS] and [SEP] tokens).
assert chunks_df.tokens.max() + 2 <= bi_encoder.max_seq_length, chunks_df.tokens.max()
print(
    f"{len(units):,} sentences -> {len(chunks_df)} chunks | tokens per chunk: "
    f"mean {chunks_df.tokens.mean():.0f}, max {chunks_df.tokens.max()} (bi-encoder limit {bi_encoder.max_seq_length})"
)
chunks_df.loc[54:56]

2,200 sentences -> 357 chunks | tokens per chunk: mean 180, max 231 (bi-encoder limit 256)


,page,text,tokens
chunk_id,,,
54,24,Passwords must be of sufficient effective strength and secrecy that it would be impractical for an attacke...,197
55,25,Passwords are not phishing-resistant. 3.1.1.1. Password Authenticators Passwords SHALL either be chosen by...,173
56,25,1. Verifiers and CSPs SHALL require passwords that are used as a single-factor authentication mechanism to...,199


## 4. Pipeline 1: bi-encoder retrieval

Every chunk is embedded once, independently of any query, with `all-MiniLM-L6-v2`. The vectors are L2-normalised, so cosine similarity equals the inner product. They are stored in an in-memory Chroma collection that uses cosine distance. At query time the question is embedded with the same model and the `TOP_K = 10` nearest chunks are returned; Pipeline 1 passes the first `TOP_N = 3` of them to the LLM.

`full_ranking` ranks *all* chunks by exact (brute-force) cosine similarity. It is used to check that Chroma's approximate HNSW index returns the exact top 10, and to report the rank of an answer chunk that falls outside the top 10.

In [4]:
t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(chunks_df.text.tolist(), batch_size=32, normalize_embeddings=True)
index_seconds = time.perf_counter() - t0

client = chromadb.EphemeralClient(settings=chromadb.Settings(anonymized_telemetry=False))
collection = client.get_or_create_collection(
    "nist-800-63b-4",
    metadata={"hnsw:space": "cosine"},
    embedding_function=None,  # vectors always come from bi_encoder, never from Chroma's default model
)
collection.upsert(
    ids=[str(i) for i in chunks_df.index],
    embeddings=chunk_embeddings,
    documents=chunks_df.text.tolist(),
    metadatas=[{"page": int(p)} for p in chunks_df.page],
)
print(f"Embedded {len(chunks_df)} chunks in {index_seconds:.1f} s; Chroma collection holds {collection.count()} vectors of dim {chunk_embeddings.shape[1]}")


def embed_query(query: str) -> np.ndarray:
    return bi_encoder.encode(query, normalize_embeddings=True)


def vector_search(query_embedding: np.ndarray, k: int = TOP_K) -> pd.DataFrame:
    # Top-k chunks from the vector store, most similar first.
    result = collection.query(query_embeddings=[query_embedding], n_results=k)
    return pd.DataFrame(
        {
            "chunk_id": [int(i) for i in result["ids"][0]],
            "bi_score": [1 - d for d in result["distances"][0]],  # Chroma returns cosine distance = 1 - similarity
            "bi_rank": range(1, k + 1),
        }
    )


def full_ranking(query_embedding: np.ndarray) -> dict[int, int]:
    # Exact rank (1 = most similar) of every chunk, by brute-force cosine similarity.
    order = np.argsort(-(chunk_embeddings @ query_embedding))
    return {int(c): rank for rank, c in enumerate(order, 1)}

Embedded 357 chunks in 11.4 s; Chroma collection holds 357 vectors of dim 384


## 5. Pipeline 2: cross-encoder re-ranking

The cross-encoder takes the query and one chunk together as a single input and outputs one relevance score (a logit; higher means more relevant). It never sees the whole collection: it scores only the 10 candidates returned by Pipeline 1. These are re-sorted by that score, and the new top 3 go to the LLM.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")


def rerank(query: str, candidates: pd.DataFrame) -> pd.DataFrame:
    # Scores every (query, chunk) pair and re-sorts the candidates by that score.
    pairs = [(query, chunks_df.text[c]) for c in candidates.chunk_id]
    reranked = candidates.assign(ce_score=cross_encoder.predict(pairs)).sort_values("ce_score", ascending=False)
    return reranked.assign(ce_rank=range(1, len(reranked) + 1))

## 6. Answer generation (identical prompt for both pipelines)

Both pipelines call the same `generate_answer` function with their own top-3 chunk ids, in their own rank order. The prompt restricts the model to the numbered passages, asks it to state any conditions or exceptions attached to a rule, and to say so when the passages do not answer the question. Temperature 0 and a fixed seed make reruns as repeatable as the API allows.

In [6]:
SYSTEM_PROMPT = (
    "You answer questions about the NIST SP 800-63B-4 guideline using ONLY the numbered context passages. "
    "State any conditions or exceptions that the passages attach to a rule. "
    "If the passages do not contain the answer, say that the context does not say. "
    "Answer in 2-4 sentences and cite passages like [1]."
)


def build_user_prompt(query: str, chunk_ids: list[int]) -> str:
    context = "\n\n".join(f"[{i}] (page {chunks_df.page[c]}) {chunks_df.text[c]}" for i, c in enumerate(chunk_ids, 1))
    return f"Context:\n{context}\n\nQuestion: {query}"


# The SDK retries rate-limited requests (HTTP 429) after the wait that Groq asks for.
llm = Groq(max_retries=6) if os.environ.get("GROQ_API_KEY") else None


def generate_answer(query: str, chunk_ids: list[int]) -> tuple[str, float]:
    # Returns (answer text, seconds taken by the API call).
    if llm is None:
        return "(skipped: set GROQ_API_KEY to generate answers)", float("nan")
    t0 = time.perf_counter()
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(query, chunk_ids)},
        ],
        temperature=0,
        seed=0,
        reasoning_effort="low",  # gpt-oss is a reasoning model; its hidden reasoning shares the token budget below
        max_completion_tokens=1024,
    )
    return response.choices[0].message.content.strip(), time.perf_counter() - t0


print("LLM:", LLM_MODEL if llm else "disabled (no GROQ_API_KEY)")

LLM: openai/gpt-oss-120b


## 7. Running both pipelines on one query

`run_both` runs Pipeline 1 and Pipeline 2 on a query, generates Answer 1 and Answer 2, and times each retrieval stage (the latency section reuses these timings). It also asserts that Chroma's top 10 is identical to the exact brute-force top 10.

In [7]:
def run_both(query: str, generate: bool = True) -> dict:
    t0 = time.perf_counter()
    query_embedding = embed_query(query)
    t1 = time.perf_counter()
    candidates = vector_search(query_embedding)  # Pipeline 1: top 10
    t2 = time.perf_counter()
    reranked = rerank(query, candidates)  # Pipeline 2: same 10, re-sorted
    t3 = time.perf_counter()

    exact = full_ranking(query_embedding)
    assert [exact[c] for c in candidates.chunk_id] == list(range(1, TOP_K + 1)), "Chroma top 10 != exact top 10"

    result = {
        "query": query,
        "candidates": reranked.sort_values("bi_rank"),  # one row per candidate with both ranks and scores
        "baseline_ids": candidates.chunk_id[:TOP_N].tolist(),
        "reranked_ids": reranked.chunk_id[:TOP_N].tolist(),
        "full_rank": exact,
        "t_embed": t1 - t0,
        "t_search": t2 - t1,
        "t_rerank": t3 - t2,
    }
    if generate:
        result["answer1"], result["t_llm1"] = generate_answer(query, result["baseline_ids"])
        result["answer2"], result["t_llm2"] = generate_answer(query, result["reranked_ids"])
    return result

## 8. Failure-mode analysis (stress tests)

Each query below targets one failure category from the assignment. The **gold chunk** is a chunk that contains the sentence answering the question. It is identified by a short quote from that sentence (`evidence`), not by chunk number, so the labels survive changes to the chunking. When overlap puts the sentence in two neighbouring chunks, both count as gold.

The queries were found by writing several candidate questions for each category and screening them with both pipelines. For many other phrasings the bi-encoder already placed the gold chunk in its top 3, so these are the clearest failures, not typical behaviour.

For each query the notebook shows:
- the 10 bi-encoder candidates with both ranks and scores, and ✓ on gold chunks,
- the rank shift of the gold chunk,
- Answer 1 (baseline top 3) and Answer 2 (re-ranked top 3).

In [8]:
STRESS_TESTS = [
    {
        "category": "Keyword trap / distractor",
        "query": "Can we lock the account after 3 failed logins?",
        "evidence": ["The limit of 100 attempts is an upper bound, and agencies MAY impose lower limits"],
    },
    {
        "category": "Exception / corner case",
        "query": "Must out-of-band secrets always be sent over an encrypted channel?",
        "evidence": ["SHALL use approved encryption unless sent via the public switched telephone network"],
    },
    {
        "category": "Negation / constraint",
        "query": "Is voice recognition allowed as a biometric for authentication?",
        "evidence": ["Biometric comparison based on voice SHALL NOT be used"],
    },
    {
        "category": "Buried normative rule (informative text outranks it)",
        "query": "What is the minimum password length?",
        "evidence": ["minimum of 15 characters in length"],
    },
    {
        "category": "Synonym trap (PIN vs. activation secret)",
        "query": "What is the minimum length of a PIN used to unlock a multi-factor authenticator?",
        "evidence": ["SHALL require the secrets to be at least four characters in length"],
    },
]


def gold_chunks(evidence: list[str]) -> list[int]:
    ids = [c for c, text in chunks_df.text.items() if any(e in text for e in evidence)]
    assert ids, f"no chunk contains any of {evidence}"
    return ids


def quote(text: str) -> str:
    return "> " + text.replace("\n", "\n> ")


def analyse(case: dict) -> dict:
    # Runs both pipelines, displays the candidate table, rank shift and both answers; returns a summary row.
    result = run_both(case["query"])
    gold = gold_chunks(case["evidence"])
    table = result["candidates"].assign(
        gold=lambda d: np.where(d.chunk_id.isin(gold), "✓", ""),
        page=lambda d: d.chunk_id.map(chunks_df.page),
        text=lambda d: d.chunk_id.map(chunks_df.text.str[:100]) + "…",
    )[["bi_rank", "ce_rank", "gold", "chunk_id", "page", "bi_score", "ce_score", "text"]]

    bi_best = min(result["full_rank"][g] for g in gold)  # rank in the whole index, defined even beyond the top 10
    in_top10 = table[table.gold == "✓"]
    if len(in_top10):
        moved = in_top10.sort_values("ce_rank").iloc[0]  # the gold chunk that the re-ranker placed highest
        ce_best = int(moved.ce_rank)
        shift = f"chunk {moved.chunk_id} (page {moved.page}) moved from bi-encoder rank #{moved.bi_rank} to cross-encoder rank #{ce_best}"
    else:
        ce_best = None
        shift = "no gold chunk in the bi-encoder top 10 ({}), so the cross-encoder never scores it".format(
            ", ".join(f"chunk {g} at rank #{result['full_rank'][g]}" for g in gold)
        )

    display(Markdown(f"### {case['category']}\n**Query:** {case['query']}"))
    display(table.set_index("bi_rank").round({"bi_score": 3, "ce_score": 2}))
    display(
        Markdown(
            f"**Rank shift:** {shift}.  \n"
            f"Gold chunk in the top {TOP_N}: baseline **{'yes' if bi_best <= TOP_N else 'no'}**, "
            f"re-ranked **{'yes' if ce_best and ce_best <= TOP_N else 'no'}**.\n\n"
            f"**Answer 1** (baseline top {TOP_N}: chunks {result['baseline_ids']}, {result['t_llm1']:.1f} s)\n\n{quote(result['answer1'])}\n\n"
            f"**Answer 2** (re-ranked top {TOP_N}: chunks {result['reranked_ids']}, {result['t_llm2']:.1f} s)\n\n{quote(result['answer2'])}"
        )
    )
    return {
        "category": case["category"],
        "query": case["query"],
        "gold_chunks": gold,
        "baseline_rank": bi_best,
        "reranked_rank": ce_best,
        "gold_in_baseline_top3": bi_best <= TOP_N,
        "gold_in_reranked_top3": bool(ce_best and ce_best <= TOP_N),
        "baseline_top3": result["baseline_ids"],
        "reranked_top3": result["reranked_ids"],
        "answer1": result["answer1"],
        "answer2": result["answer2"],
        "t_llm1": result["t_llm1"],
        "t_llm2": result["t_llm2"],
        "candidates": table.assign(query=case["query"]),
    }

In [9]:
stress_0 = analyse(STRESS_TESTS[0])

### Keyword trap / distractor
**Query:** Can we lock the account after 3 failed logins?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,2,,165,58,0.457,-9.18,"Once this is completed, the subscriber can bind one or more new authenticators to their subscriber a…"
2,6,,176,61,0.410,-10.56,"Generally, one must assume that a lost authenticator has been stolen or compromised by someone other…"
3,5,,174,60,0.392,-10.47,"Recovery at AAL2 To recover an account that can authenticate at a maximum of AAL2, the CSP SHALL req…"
4,4,,175,61,0.387,-10.20,"If an account that can authenticate at AAL3 has been identity-proofed at IAL3, the CSP SHALL success…"
5,1,✓,110,41,0.387,-7.87,"Rate Limiting (Throttling) When required by the authenticator type descriptions in Sec. 3.1, the ver…"
6,3,,177,61,0.386,-9.46,"Alternatively, the subscriber MAY establish an authenticated protected channel for the CSP to verify…"
7,9,,214,76,0.385,-11.04,6.3. Authenticator Recovery The weak point in many authentication mechanisms is the process followed…
8,7,,39,18,0.378,-10.70,"2.2.3. Reauthentication Periodic reauthentication of subscriber sessions SHALL be performed, as desc…"
9,10,,137,49,0.375,-11.13,Offer subscribers at least one alternative authenticator that is not restricted and can be used to a…


**Rank shift:** chunk 110 (page 41) moved from bi-encoder rank #5 to cross-encoder rank #1.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [165, 176, 174], 0.6 s)

> The provided passages do not address a policy of locking an account after a certain number of failed login attempts. Therefore, the context does not say whether an account can be locked after three failed logins.

**Answer 2** (re-ranked top 3: chunks [110, 165, 177], 0.5 s)

> Yes. The guideline requires a verifier to limit consecutive failed attempts, but the 100‑attempt ceiling is only an upper bound; agencies are permitted to set lower limits. Therefore an implementation could lock an account after 3 failed logins. [1]

In [10]:
stress_1 = analyse(STRESS_TESTS[1])

### Exception / corner case
**Query:** Must out-of-band secrets always be sent over an encrypted channel?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,5,,73,31,0.612,3.60,This has been observed with “authentication fatigue” attacks in which an attacker generates many out…
2,2,,77,32,0.567,3.98,"For the purposes of this definition, “wired connection” includes services such as cable providers th…"
3,10,,187,65,0.538,-2.57,"2. Secrets are established using input from an approved random bit generator, as described in Sec. 3…"
4,4,,69,29,0.531,3.60,Both the salt value and the resulting hash SHALL be stored for each look-up secret. • The verifier S…
5,3,,76,32,0.529,3.91,Communication between the out-of-band authenticator and the verifier MAY use a trusted intermediary …
6,9,,79,32,0.528,1.95,"Depending on the type of out-of-band authenticator, one of the following SHALL take place: • Transfe…"
7,1,✓,74,31,0.524,6.22,Out-of-Band Authenticators The out-of-band authenticator SHALL establish a separate channel with the…
8,8,,70,29,0.522,2.35,The device is possessed and controlled by the claimant and supports private communication over this …
9,7,,71,29,0.517,2.75,Out-of-band authentication is not phishing-resistant. The out-of-band authenticator can operate in o…


**Rank shift:** chunk 74 (page 31) moved from bi-encoder rank #7 to cross-encoder rank #1.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [73, 77, 187], 0.4 s)

> Yes. The guideline requires that out‑of‑band secrets be transferred only over an authenticated protected channel (i.e., an encrypted channel) rather than in the clear. This is stated in the secret‑handling rules, which say that “Secrets are either transferred … via an authenticated protected channel” [3].

**Answer 2** (re-ranked top 3: chunks [74, 77, 76], 0.8 s)

> No. The guideline requires that the out‑of‑band channel use approved encryption **unless** the secret is sent over the public switched telephone network (PSTN); in that case encryption is not required. [1]

In [11]:
stress_2 = analyse(STRESS_TESTS[2])

### Negation / constraint
**Query:** Is voice recognition allowed as a biometric for authentication?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,6,,116,43,0.531,-0.34,"However, the availability of such solutions is limited. • Biometric characteristics do not constitut…"
2,3,,321,115,0.525,0.27,The ability to generate valid authenticator outputs on demand proves that the claimant possesses and…
3,7,,115,42,0.511,-0.74,The chosen threshold usually emphasizes a low FMR to maximize security since false non-matches can o…
4,1,,113,42,0.495,3.19,"If this is provided, the maximum AAL of the authenticator being reset SHALL not exceed the AAL of th…"
5,2,✓,119,43,0.486,1.94,Biometric comparison based on voice SHALL NOT be used. Testing the biometric system for deployment S…
6,4,,252,90,0.477,0.11,• An alternative authentication method must be readily available and clearly communicated. Users sho…
7,10,,134,48,0.475,-1.67,"Authentication processes that require the claimant’s intervention can be used to prove intent (e.g.,…"
8,5,,117,43,0.465,-0.24,Biometrics SHALL only be used as part of multi-factor authentication with a physical authenticator (…
9,9,✓,118,43,0.464,-1.14,The biometric system SHOULD demonstrate a false non-match rate (FNMR) of less than 5 %. Biometric pe…


**Rank shift:** chunk 119 (page 43) moved from bi-encoder rank #5 to cross-encoder rank #2.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [116, 321, 115], 0.8 s)

> Yes. Voice recognition (voice prints) is listed as a biometric characteristic in the guideline [2]. However, the guideline permits biometrics only when they are used as part of multi‑factor authentication that includes a physical authenticator (“something you have”) [1]. So voice‑based biometrics can be used, but only as a secondary factor combined with a physical token.

**Answer 2** (re-ranked top 3: chunks [113, 119, 321], 0.7 s)

> No. NIST SP 800‑63B‑4 explicitly prohibits using voice‑based biometric comparisons for authentication; the guideline states that “Biometric comparison based on voice SHALL NOT be used”【2】.

In [12]:
stress_3 = analyse(STRESS_TESTS[3])

### Buried normative rule (informative text outranks it)
**Query:** What is the minimum password length?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,2,,284,98,0.709,6.30,These attacks are outside of the scope of this Appendix. A.2. Length Password length is a primary fa…
2,5,,236,83,0.651,0.64,• When users create and change passwords – Clearly communicate information on how to create and chan…
3,6,,286,99,0.623,-0.73,"Since the size of a hashed password is independent of its length, there is no reason to prohibit the…"
4,7,,283,98,0.605,-2.53,"However, analyses of breached password databases reveal that the benefit of such rules is less signi…"
5,1,✓,56,25,0.565,6.69,1. Verifiers and CSPs SHALL require passwords that are used as a single-factor authentication mechan…
6,3,✓,55,25,0.533,1.86,Passwords are not phishing-resistant. 3.1.1.1. Password Authenticators Passwords SHALL either be cho…
7,10,,256,92,0.511,-10.26,"23043 [CMVP] National Institute of Standards and Technology (2025), Cryptographic Module Validation …"
8,8,,285,98,0.509,-4.31,3.2.2 can be applied before there is a significant chance of a successful guess. Offline attacks are…
9,4,,288,99,0.501,1.48,"These include dictionary words and passwords from previous breaches, such as the “Password1!” exampl…"


**Rank shift:** chunk 56 (page 25) moved from bi-encoder rank #5 to cross-encoder rank #1.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [284, 236, 286], 0.5 s)

> The guideline does not prescribe a single numeric minimum; it states that “the minimum password length required depends on the threat model being addressed” [1]. Thus, the required length is determined by the specific risks being mitigated rather than a fixed value.

**Answer 2** (re-ranked top 3: chunks [56, 284, 55], 0.4 s)

> The guideline requires that passwords used as a single‑factor authenticator be at least 15 characters long, while passwords that are only part of a multi‑factor process may be shorter but must be at least 8 characters long [1]. This same 15‑character minimum for single‑factor passwords is reiterated in the password verifier requirements [3].

In [13]:
stress_4 = analyse(STRESS_TESTS[4])

### Synonym trap (PIN vs. activation secret)
**Query:** What is the minimum length of a PIN used to unlock a multi-factor authenticator?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,1,,101,38,0.566,1.91,The challenge nonce SHALL be at least 64 bits in length and SHALL either be unique over the authenti…
2,7,,104,39,0.564,-1.74,Authenticator activation secrets SHALL meet the requirements of Sec. 3.2.10. A biometric activation …
3,4,,35,17,0.554,-0.80,"2.2.1. Permitted Authenticator Types At AAL2, authentication SHALL use either a multi-factor authent…"
4,2,✓,138,49,0.553,1.14,"An activation secret is used to obtain access to a stored authentication key. In all cases, the acti…"
5,3,,53,23,0.517,-0.19,"prove possession of the authenticator 3.1.4, 3.1.5 Activation secret A password that is used locally…"
6,6,,240,85,0.513,-1.25,• Authenticator output allows at least one minute between changes but ideally allows users two full …
7,8,,137,49,0.499,-2.68,Offer subscribers at least one alternative authenticator that is not restricted and can be used to a…
8,5,,102,39,0.491,-1.11,The authenticator output is highly dependent on the specific cryptographic protocol used but is gene…
9,9,,84,34,0.482,-5.34,"3.1.3.1). However, they require the presentation and verification of an activation factor (i.e., a p…"


**Rank shift:** chunk 138 (page 49) moved from bi-encoder rank #4 to cross-encoder rank #2.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [101, 104, 35], 0.6 s)

> The provided passages do not specify a minimum length for a PIN used to unlock a multi‑factor authenticator. None of the cited sections mention PIN length requirements. Therefore, the context does not say.

**Answer 2** (re-ranked top 3: chunks [101, 138, 53], 0.4 s)

> The guideline requires that an activation secret (which may be a numeric PIN) be at least four characters long, though a six‑character length is recommended. [2]

### Rank-shift summary

`baseline_rank` is the best rank of a gold chunk in the bi-encoder ranking; `reranked_rank` is its best rank after the cross-encoder re-sorts the top 10.

In [14]:
stress_rows = [stress_0, stress_1, stress_2, stress_3, stress_4]
SUMMARY_COLUMNS = ["category", "query", "gold_chunks", "baseline_rank", "reranked_rank", "gold_in_baseline_top3", "gold_in_reranked_top3"]
pd.DataFrame(stress_rows, columns=SUMMARY_COLUMNS)

,category,query,gold_chunks,baseline_rank,reranked_rank,gold_in_baseline_top3,gold_in_reranked_top3
0,Keyword trap / distractor,Can we lock the account after 3 failed logins?,"[110, 111]",5,1,False,True
1,Exception / corner case,Must out-of-band secrets always be sent over an encrypted channel?,[74],7,1,False,True
2,Negation / constraint,Is voice recognition allowed as a biometric for authentication?,"[118, 119]",5,2,False,True
3,Buried normative rule (informative text outranks it),What is the minimum password length?,"[55, 56]",5,1,False,True
4,Synonym trap (PIN vs. activation secret),What is the minimum length of a PIN used to unlock a multi-factor authenticator?,[138],4,2,False,True


## 9. Cases that re-ranking did not fix

Two stress queries for which the cross-encoder was not enough:

- **Recall ceiling.** The guideline's restriction on SMS codes is written in terms of out-of-band authentication over the "PSTN" (public switched telephone network). If no chunk stating that restriction reaches the bi-encoder's top 10, the cross-encoder cannot recover it, because it only re-orders those 10.
- **Exception promoted, but is it used?** The general rule and its exception sit in neighbouring chunks. Re-ranking promotes the exception chunk into the top 3; compare whether Answer 2 actually uses it.

In [15]:
HARD_CASES = [
    {
        "category": "Recall ceiling (SMS vs. PSTN terminology)",
        "query": "Is SMS two-factor authentication still allowed?",
        "evidence": ["Use of the PSTN for out-of-band verification is restricted", "there is one restricted authenticator"],
    },
    {
        "category": "Exception promoted into the top 3",
        "query": "Can session secrets persist after the device is rebooted?",
        "evidence": ["Session secrets used with such proof of possession techniques MAY persist"],
    },
]
hard_0 = analyse(HARD_CASES[0])

### Recall ceiling (SMS vs. PSTN terminology)
**Query:** Is SMS two-factor authentication still allowed?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,1,,35,17,0.552,0.13,"2.2.1. Permitted Authenticator Types At AAL2, authentication SHALL use either a multi-factor authent…"
2,10,,37,18,0.518,-7.44,"As noted in that section, local verification of biometric factors (i.e., the use of a multi-factor a…"
3,3,,91,36,0.489,-3.51,This includes hardware devices and software-based OTP generators that are installed on mobile phones…
4,9,,106,40,0.486,-6.83,The assertion contents and presentation requirements of the federation protocol provide the security…
5,7,,332,120,0.485,-6.19,MACs provide authenticity and integrity protection but not non-repudiation protection. mobile code E…
6,8,,104,39,0.480,-6.39,Authenticator activation secrets SHALL meet the requirements of Sec. 3.2.10. A biometric activation …
7,2,,239,84,0.471,-2.61,"For example, it is difficult for users to transfer the authentication secret manually using a smartp…"
8,4,,105,39,0.469,-4.01,"If such a flag is present and indicates that an activation factor was not used, the authentication S…"
9,5,,22,12,0.468,-4.60,Authentication at higher AALs can effectively reduce the risk of attacks. A high-level summary of th…


**Rank shift:** no gold chunk in the bi-encoder top 10 (chunk 82 at rank #72, chunk 135 at rank #60), so the cross-encoder never scores it.  
Gold chunk in the top 3: baseline **no**, re-ranked **no**.

**Answer 1** (baseline top 3: chunks [35, 37, 91], 0.6 s)

> The provided passages do not address whether SMS‑based two‑factor authentication is permitted or prohibited under AAL2. Therefore, the context does not say.

**Answer 2** (re-ranked top 3: chunks [35, 239, 91], 1.6 s)

> The provided passages do not address whether SMS‑based two‑factor authentication is permitted or prohibited. Therefore, the context does not say.

In [16]:
hard_1 = analyse(HARD_CASES[1])

### Exception promoted into the top 3
**Query:** Can session secrets persist after the device is rebooted?

,ce_rank,gold,chunk_id,page,bi_score,ce_score,text
bi_rank,,,,,,,
1,1,,185,64,0.691,6.73,"The nature of a session depends on the application, such as: • A web browser session with a “session…"
2,3,,188,65,0.574,1.47,"In addition, the secrets used for session binding SHOULD be erased on the subscriber endpoint when t…"
3,6,,184,64,0.567,-1.42,"5.1. Session Bindings A session occurs between the software (i.e., the session subject) that a subsc…"
4,2,✓,186,65,0.549,3.02,These technologies might additionally store and use these session secrets in protected keystores to …
5,4,,187,65,0.506,0.16,"2. Secrets are established using input from an approved random bit generator, as described in Sec. 3…"
6,8,,183,64,0.482,-3.96,This is particularly the case with federation scenarios (see [SP800-63C]) in which the authenticatio…
7,5,,195,67,0.474,-0.61,"Likewise, the subscriber’s sessions at multiple different RPs are established and terminated indepen…"
8,9,,193,67,0.468,-4.58,"Session activity SHALL reset the inactivity timeout, and successful reauthentication during a sessio…"
9,7,,215,76,0.452,-1.74,6.4. Session Attacks Hijacking attacks on the session following an authentication event can have sim…


**Rank shift:** chunk 186 (page 65) moved from bi-encoder rank #4 to cross-encoder rank #2.  
Gold chunk in the top 3: baseline **no**, re-ranked **yes**.

**Answer 1** (baseline top 3: chunks [185, 188, 184], 4.7 s)

> Session secrets that are used as bearer tokens for session management SHOULD NOT be persistent—that is, they must not be retained across a restart of the associated application or a reboot of the host device, because a restart or reboot ends the specific session. [1]

**Answer 2** (re-ranked top 3: chunks [185, 186, 188], 6.8 s)

> The guideline says that session secrets used as bearer tokens **should not be persistent**—they must not be retained across a restart of the application or a reboot of the host device because those events end the specific session [1]. Therefore, session secrets are not allowed to persist after a device reboot.

## 10. Latency profiling

**Retrieval stages.** Each of the 7 queries above is run once as a warm-up and then `REPEATS = 20` times, timing query embedding, Chroma search (top 10) and cross-encoder scoring (10 pairs) separately. Pipeline 1's retrieval is embedding + search; Pipeline 2 adds the cross-encoder on top. The table shows the median and 95th percentile over all 140 timed runs, in milliseconds.

**LLM.** One Groq call per answer, timed in sections 8–9. These times include the network round trip and any wait imposed by the free tier's rate limit, and the model and prompt template are the same for both pipelines.

In [17]:
REPEATS = 20
profile_queries = [case["query"] for case in STRESS_TESTS + HARD_CASES]
for q in profile_queries:  # warm-up
    run_both(q, generate=False)

timings = pd.DataFrame(
    [
        {k: v * 1000 for k, v in run_both(q, generate=False).items() if k.startswith("t_")}
        for q in profile_queries
        for _ in range(REPEATS)
    ]
)
all_rows = stress_rows + [hard_0, hard_1]
llm1 = pd.Series([r["t_llm1"] * 1000 for r in all_rows])
llm2 = pd.Series([r["t_llm2"] * 1000 for r in all_rows])

stages = {
    "Query embedding (bi-encoder)": (timings.t_embed, timings.t_embed),
    "Vector search, top 10 (Chroma)": (timings.t_search, timings.t_search),
    "Cross-encoder, 10 pairs": (None, timings.t_rerank),
    "Retrieval total": (timings.t_embed + timings.t_search, timings.t_embed + timings.t_search + timings.t_rerank),
    "LLM answer (Groq API)": (llm1, llm2),
}
latency = pd.DataFrame(
    {
        stage: {
            "Pipeline 1 median (ms)": p1.median() if p1 is not None else np.nan,
            "Pipeline 1 p95 (ms)": p1.quantile(0.95) if p1 is not None else np.nan,
            "Pipeline 2 median (ms)": p2.median(),
            "Pipeline 2 p95 (ms)": p2.quantile(0.95),
        }
        for stage, (p1, p2) in stages.items()
    }
).T
latency.loc["End-to-end (median retrieval + median LLM)"] = [
    latency.loc["Retrieval total", "Pipeline 1 median (ms)"] + llm1.median(), np.nan,
    latency.loc["Retrieval total", "Pipeline 2 median (ms)"] + llm2.median(), np.nan,
]
overhead = latency.loc["Retrieval total", "Pipeline 2 median (ms)"] / latency.loc["Retrieval total", "Pipeline 1 median (ms)"]
print(f"{len(timings)} timed runs per stage on {cpu_name()}; Pipeline 2 retrieval takes {overhead:.1f}x as long as Pipeline 1 (median)")
latency.round(1)

140 timed runs per stage on 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz; Pipeline 2 retrieval takes 26.1x as long as Pipeline 1 (median)


,Pipeline 1 median (ms),Pipeline 1 p95 (ms),Pipeline 2 median (ms),Pipeline 2 p95 (ms)
Query embedding (bi-encoder),10.9,15.1,10.9,15.1
"Vector search, top 10 (Chroma)",2.4,5.0,2.4,5.0
"Cross-encoder, 10 pairs",NaN,NaN,335.1,480.8
Retrieval total,13.4,19.0,349.8,499.7
LLM answer (Groq API),577.2,3519.0,669.1,5262.4
End-to-end (median retrieval + median LLM),590.6,NaN,1018.9,NaN


### Cost of running the cross-encoder over the whole collection

For comparison, the cross-encoder scores *every* chunk for one query (median of 3 runs). Unlike the bi-encoder, nothing can be pre-computed at indexing time, because each score depends on the query. The cross-encoder rows for one million chunks extrapolate the measured time per pair linearly.

For the bi-encoder at the same scale, an exact top-10 inner-product search over 100,000 random 384-dim vectors is timed and multiplied by 10 (one million vectors would need about 1.5 GB of RAM). The query embedding time is added to it. The chunk vectors themselves are computed once, at indexing time.

In [18]:
def median_seconds(fn, runs: int = 3) -> float:
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


query = STRESS_TESTS[0]["query"]
query_embedding = embed_query(query)
all_pairs = [(query, text) for text in chunks_df.text]
ce_all_seconds = median_seconds(lambda: cross_encoder.predict(all_pairs, batch_size=32))
per_pair_ms = ce_all_seconds / len(all_pairs) * 1000

random_vectors = np.random.default_rng(0).standard_normal((100_000, chunk_embeddings.shape[1]), dtype=np.float32)
search_100k_seconds = median_seconds(lambda: np.argpartition(-(random_vectors @ query_embedding), TOP_K)[:TOP_K])
del random_vectors

embed_s = latency.loc["Query embedding (bi-encoder)", "Pipeline 1 median (ms)"] / 1000
scaling = pd.DataFrame(
    [
        (f"Bi-encoder: embed query + Chroma top 10 over {len(chunks_df)} chunks (Pipeline 1)", len(chunks_df),
         latency.loc["Retrieval total", "Pipeline 1 median (ms)"] / 1000),
        ("Bi-encoder: embed query + exact top 10 over 1,000,000 vectors (extrapolated)", 1_000_000,
         embed_s + 10 * search_100k_seconds),
        ("Cross-encoder: re-rank the top 10 (Pipeline 2 overhead)", 10,
         latency.loc["Cross-encoder, 10 pairs", "Pipeline 2 median (ms)"] / 1000),
        (f"Cross-encoder: all {len(chunks_df)} chunks of this PDF", len(chunks_df), ce_all_seconds),
        ("Cross-encoder: 1,000,000 chunks (extrapolated)", 1_000_000, per_pair_ms * 1_000_000 / 1000),
    ],
    columns=["approach", "items scored per query", "seconds per query"],
)
print(
    f"Cross-encoder: {per_pair_ms:.1f} ms per (query, chunk) pair when batched, i.e. about "
    f"{per_pair_ms * 1_000_000 / 3_600_000:.1f} hours per query for one million chunks. "
    f"One-off bi-encoder indexing of all {len(chunks_df)} chunks took {index_seconds:.1f} s."
)
scaling.round(3)

Cross-encoder: 30.4 ms per (query, chunk) pair when batched, i.e. about 8.4 hours per query for one million chunks. One-off bi-encoder indexing of all 357 chunks took 11.4 s.


,approach,items scored per query,seconds per query
0,Bi-encoder: embed query + Chroma top 10 over 357 chunks (Pipeline 1),357,0.013
1,"Bi-encoder: embed query + exact top 10 over 1,000,000 vectors (extrapolated)",1000000,0.079
2,Cross-encoder: re-rank the top 10 (Pipeline 2 overhead),10,0.335
3,Cross-encoder: all 357 chunks of this PDF,357,10.841
4,"Cross-encoder: 1,000,000 chunks (extrapolated)",1000000,30367.561


## 11. Saved results

The tables above are written to `results/` as CSV files:

- `rank_shifts.csv`: one row per stress query (gold chunk, ranks before and after re-ranking, top-3 hit or miss)
- `candidates.csv`: the full top-10 table of every stress query (both ranks and scores)
- `answers.csv`: Answer 1 and Answer 2 with the chunk ids each one was given
- `latency.csv` and `scaling.csv`: the latency profile

In [19]:
ANSWER_COLUMNS = ["category", "query", "baseline_top3", "answer1", "reranked_top3", "answer2"]
pd.DataFrame(all_rows, columns=SUMMARY_COLUMNS).astype({"reranked_rank": "Int64"}).to_csv(  # Int64 keeps "no rank" empty
    RESULTS_DIR / "rank_shifts.csv", index=False
)
pd.DataFrame(all_rows, columns=ANSWER_COLUMNS).to_csv(RESULTS_DIR / "answers.csv", index=False)
pd.concat([r["candidates"] for r in all_rows]).to_csv(RESULTS_DIR / "candidates.csv", index=False)
latency.round(2).to_csv(RESULTS_DIR / "latency.csv", index_label="stage")
scaling.to_csv(RESULTS_DIR / "scaling.csv", index=False)
sorted(p.name for p in RESULTS_DIR.iterdir())

['answers.csv',
 'candidates.csv',
 'latency.csv',
 'rank_shifts.csv',
 'scaling.csv']